In [ ]:
# -*- coding: utf-8 -*-
"""
Ultimate Optimized Pipeline for Feedback Prize: Evaluating Student Writing
Model: microsoft/deberta-v3-base
Features:
  - Dynamic Padding & Length Grouping (~5x Training Speedup)
  - Last-4-Layers Mean Pooling + BiLSTM + Multi-Sample Dropout
  - Layer-wise Learning Rate Decay (LLRD) + Cosine Schedule
  - Word-level Subword Mean Probability Aggregation
  - Fast Validation & Advanced Multi-stage Post-Processing
"""

import os
import gc
import re
import math
import time
import random
from pathlib import Path
from typing import List, Dict, Tuple, Any, Optional, Set

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from sklearn.model_selection import train_test_split

from transformers import AutoTokenizer, AutoConfig, AutoModel

# -------------------------------------------------------------
# 1. Configuration & Reproducibility
# -------------------------------------------------------------
def seed_everything(seed: int = 42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Active Device: {device}")

CFG = {
    "model_name": "microsoft/deberta-v3-base",
    "max_len": 1024,
    "epochs": 3,
    "train_batch_size": 4,           
    "gradient_accumulation_steps": 4,   # Effective batch size = 16
    "encoder_lr": 2.0e-5,
    "head_lr": 1.0e-4,
    "llrd_decay": 0.90,
    "weight_decay": 0.01,
    "warmup_ratio": 0.10,
    "max_grad_norm": 1.0,
    "label_smoothing": 0.03,
    "lstm_hidden_size": 512,
    "msd_dropouts": [0.1, 0.2, 0.3, 0.4, 0.5],
    "fast_val_size": 400,              
    "output_dir": "outputs_deberta_optimized/",
    "checkpoint_path": "outputs_deberta_optimized/best_deberta_v3_base.pt"
}

os.makedirs(CFG["output_dir"], exist_ok=True)

DISCOURSE_TYPES = [
    "Lead",
    "Position",
    "Claim",
    "Counterclaim",
    "Rebuttal",
    "Evidence",
    "Concluding Statement"
]

LABELS = ["O"]
for dt in DISCOURSE_TYPES:
    LABELS.append(f"B-{dt}")
    LABELS.append(f"I-{dt}")

LABEL2ID = {lbl: i for i, lbl in enumerate(LABELS)}
ID2LABEL = {i: lbl for i, lbl in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

# EDA-Calibrated Minimum Span Lengths
EDA_MIN_LENGTHS = {
    'Evidence': 21,
    'Concluding Statement': 14,
    'Lead': 14,
    'Position': 6,
    'Claim': 4,
    'Counterclaim': 10,
    'Rebuttal': 4
}

# Probability Decision Thresholds
TUNED_CLASS_THRESHOLDS = {
    'O': 0.64,
    'Evidence': 0.56,
    'Lead': 0.28,
    'Concluding Statement': 0.32,
    'Position': 0.28,
    'Claim': 0.36,
    'Counterclaim': 0.26,
    'Rebuttal': 0.20
}

# -------------------------------------------------------------
# 2. Dataset Resolver & Pre-tokenization
# -------------------------------------------------------------
def find_kaggle_dataset_paths() -> Tuple[Path, Path]:
    csv_candidates = [
        Path("../data/raw/train.csv"),
        Path("data/raw/train.csv"),
        Path("data/train.csv"),
        Path("train.csv"),
        Path("/content/data/raw/train.csv"),
        Path("/content/train.csv")
    ]
    txt_dir_candidates = [
        Path("../data/raw/train"),
        Path("data/raw/train"),
        Path("data/train"),
        Path("train"),
        Path("/content/data/raw/train"),
        Path("/content/train")
    ]
    found_csv = next((p.resolve() for p in csv_candidates if p.is_file()), None)
    found_txt_dir = next((p.resolve() for p in txt_dir_candidates if p.is_dir() and len(os.listdir(p)) > 0), None)

    if found_csv is None or found_txt_dir is None:
        raise FileNotFoundError("train.csv կամ train/ պանակը չգտնվեց:")
    return found_csv, found_txt_dir

CSV_PATH, TXT_DIR = find_kaggle_dataset_paths()
print(f"📊 Loading CSV: {CSV_PATH}")
df = pd.read_csv(CSV_PATH)
grouped_df = dict(tuple(df.groupby("id")))

def get_essay_text(essay_id: str, txt_dir: Path) -> str:
    with open(os.path.join(str(txt_dir), f"{essay_id}.txt"), "r", encoding="utf-8") as f:
        return f.read()

def build_word_labels(essay_id: str, text: str, grouped_df: dict) -> Tuple[List[str], List[str]]:
    words = text.split()
    word_labels = ["O"] * len(words)

    if essay_id in grouped_df:
        essay_rows = grouped_df[essay_id]
        for _, row in essay_rows.iterrows():
            dtype = row["discourse_type"]
            w_indices = [int(x) for x in str(row["predictionstring"]).split() if x.isdigit()]
            if not w_indices:
                continue

            if w_indices[0] < len(word_labels):
                word_labels[w_indices[0]] = f"B-{dtype}"

            for idx in w_indices[1:]:
                if idx < len(word_labels):
                    word_labels[idx] = f"I-{dtype}"

    return words, word_labels

tokenizer = AutoTokenizer.from_pretrained(CFG["model_name"], use_fast=True)
all_ids = df["id"].unique().tolist()
train_ids, val_ids = train_test_split(all_ids, test_size=0.15, random_state=42)

print("\n⏳թոքենիզացնում ենք բոլոր էսսեները")

def prepare_all_features(essay_ids: List[str], df: pd.DataFrame, tokenizer, txt_dir: Path, max_len: int = 1024):
    features = []
    g_df = dict(tuple(df[df["id"].isin(set(essay_ids))].groupby("id")))

    for eid in tqdm(essay_ids, desc="Tokenizing"):
        text = get_essay_text(eid, txt_dir)
        words, word_labels = build_word_labels(eid, text, g_df)

        # Դինամիկ ենք թողնում՝ առանց static max_length padding-ի
        encoding = tokenizer(
            words,
            is_split_into_words=True,
            truncation=True,
            max_length=max_len,
            padding=False,
            return_tensors=None
        )

        word_ids = encoding.word_ids()
        aligned_labels = []
        for w_id in word_ids:
            if w_id is None:
                aligned_labels.append(-100)
            else:
                label_name = word_labels[w_id] if w_id < len(word_labels) else "O"
                aligned_labels.append(LABEL2ID[label_name])

        features.append({
            "input_ids": torch.tensor(encoding["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoding["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(aligned_labels, dtype=torch.long),
            "length": len(encoding["input_ids"])
        })
    return features

train_features = prepare_all_features(train_ids, df, tokenizer, TXT_DIR, max_len=CFG["max_len"])

train_features.sort(key=lambda x: x["length"])

# -------------------------------------------------------------
# 3. Dynamic Padding DataLoader
# -------------------------------------------------------------
class FastDynamicCollator:
    def __init__(self, pad_token_id=0, label_pad_id=-100):
        self.pad_token_id = pad_token_id
        self.label_pad_id = label_pad_id

    def __call__(self, batch):
        input_ids = [item["input_ids"] for item in batch]
        attention_mask = [item["attention_mask"] for item in batch]
        labels = [item["labels"] for item in batch]

        input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=self.pad_token_id)
        attention_mask_padded = pad_sequence(attention_mask, batch_first=True, padding_value=0)
        labels_padded = pad_sequence(labels, batch_first=True, padding_value=self.label_pad_id)

        return {
            "input_ids": input_ids_padded,
            "attention_mask": attention_mask_padded,
            "labels": labels_padded
        }

collator = FastDynamicCollator(pad_token_id=tokenizer.pad_token_id or 0)

train_loader = DataLoader(
    train_features,
    batch_size=CFG["train_batch_size"],
    shuffle=True,
    collate_fn=collator,
    pin_memory=True
)

# -------------------------------------------------------------
# 4. Model Architecture (Last 4 Pooling + BiLSTM + 5-MSD)
# -------------------------------------------------------------
class FeedbackCustomDeberta(nn.Module):
    def __init__(self, model_name: str, num_labels: int, lstm_hidden: int = 512, dropouts: List[float] = None):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({"output_hidden_states": True, "hidden_dropout_prob": 0.0, "attention_probs_dropout_prob": 0.0})

        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        hidden_size = self.config.hidden_size

        self.bilstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden,
            num_layers=1,
            bidirectional=True,
            batch_first=True
        )

        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in (dropouts or [0.1, 0.2, 0.3, 0.4, 0.5])])
        self.classifier = nn.Linear(lstm_hidden * 2, num_labels)
        self._init_weights(self.classifier)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)

        # Mean Pooling over the last 4 hidden layers
        all_hidden_states = outputs.hidden_states
        hidden_states = torch.stack(all_hidden_states[-4:], dim=0).mean(dim=0)

        # BiLSTM Context Modeling
        lstm_out, _ = self.bilstm(hidden_states)

        # Multi-Sample Dropout (MSD)
        logits = torch.mean(
            torch.stack([self.classifier(dropout(lstm_out)) for dropout in self.dropouts], dim=0),
            dim=0
        )
        return logits

model = FeedbackCustomDeberta(
    CFG["model_name"],
    num_labels=NUM_LABELS,
    lstm_hidden=CFG["lstm_hidden_size"],
    dropouts=CFG["msd_dropouts"]
).to(device)

model.float()
for p in model.parameters():
    p.data = p.data.float()

# -------------------------------------------------------------
# 5. Layer-wise LR Decay & Cosine Optimizer
# -------------------------------------------------------------
def get_optimizer_grouped_parameters(model: nn.Module, encoder_lr: float, head_lr: float, weight_decay: float, llrd_decay: float = 0.90):
    no_decay = ["bias", "LayerNorm.weight", "LayerNorm.bias"]
    optimizer_parameters = []

    # Custom Head (BiLSTM + Classifier)
    head_params = list(model.bilstm.named_parameters()) + list(model.classifier.named_parameters())
    optimizer_parameters.extend([
        {"params": [p for n, p in head_params if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay, "lr": head_lr},
        {"params": [p for n, p in head_params if any(nd in n for nd in no_decay)], "weight_decay": 0.0, "lr": head_lr},
    ])

    # Embeddings
    embed_params = list(model.transformer.embeddings.named_parameters())
    optimizer_parameters.extend([
        {"params": [p for n, p in embed_params if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay, "lr": encoder_lr * (llrd_decay ** 12)},
        {"params": [p for n, p in embed_params if any(nd in n for nd in no_decay)], "weight_decay": 0.0, "lr": encoder_lr * (llrd_decay ** 12)},
    ])

    # 12 Encoder Layers
    layers = model.transformer.encoder.layer
    num_layers = len(layers)
    for i, layer in enumerate(layers):
        layer_lr = encoder_lr * (llrd_decay ** (num_layers - 1 - i))
        layer_params = list(layer.named_parameters())
        optimizer_parameters.extend([
            {"params": [p for n, p in layer_params if not any(nd in n for nd in no_decay)], "weight_decay": weight_decay, "lr": layer_lr},
            {"params": [p for n, p in layer_params if any(nd in n for nd in no_decay)], "weight_decay": 0.0, "lr": layer_lr},
        ])

    return optimizer_parameters

def get_cosine_schedule_with_warmup(optimizer, num_warmup_steps: int, num_training_steps: int):
    def lr_lambda(current_step: int):
        if current_step < num_warmup_steps:
            return float(current_step) / float(max(1, num_warmup_steps))
        progress = float(current_step - num_warmup_steps) / float(max(1, num_training_steps - num_warmup_steps))
        return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))
    return LambdaLR(optimizer, lr_lambda)

optimizer_parameters = get_optimizer_grouped_parameters(
    model,
    encoder_lr=CFG["encoder_lr"],
    head_lr=CFG["head_lr"],
    weight_decay=CFG["weight_decay"],
    llrd_decay=CFG["llrd_decay"]
)
optimizer = AdamW(optimizer_parameters)

epochs = CFG["epochs"]
grad_accum = CFG["gradient_accumulation_steps"]
total_steps = (len(train_loader) // grad_accum) * epochs
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * CFG["warmup_ratio"]),
    num_training_steps=total_steps
)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bf16 else torch.float16
use_scaler = torch.cuda.is_available() and (not use_bf16)
scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)

print(f"✅ Precision: {'bfloat16 (Scaler off)' if use_bf16 else 'float16 (Scaler on)'}")
loss_fn = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"], ignore_index=-100)

# -------------------------------------------------------------
# 6. Competition Official Metric (IoU >= 0.5 Macro-F1)
# -------------------------------------------------------------
def compute_overlap_ratio(pred_span: Tuple[int, int], true_span: Tuple[int, int]) -> float:
    p_start, p_end = pred_span
    t_start, t_end = true_span
    intersection = max(0, min(p_end, t_end) - max(p_start, t_start) + 1)
    union = max(p_end, t_end) - min(p_start, t_start) + 1
    return intersection / union if union > 0 else 0.0

def score_feedback_comp(pred_df: pd.DataFrame, gt_df: pd.DataFrame, overlap_threshold: float = 0.5) -> Tuple[float, pd.DataFrame]:
    class_metrics = []
    f1_scores = []

    for discourse_type in DISCOURSE_TYPES:
        pred_type_df = pred_df[pred_df["discourse_type"] == discourse_type] if not pred_df.empty else pd.DataFrame()
        gt_type_df = gt_df[gt_df["discourse_type"] == discourse_type] if not gt_df.empty else pd.DataFrame()

        pred_grouped = pred_type_df.groupby("id") if not pred_type_df.empty else None
        gt_grouped = gt_type_df.groupby("id") if not gt_type_df.empty else None

        all_essay_ids = set(gt_df["id"].unique()).union(set(pred_df["id"].unique()) if not pred_df.empty else set())
        tp, fp, fn = 0, 0, 0

        for essay_id in all_essay_ids:
            preds = pred_grouped.get_group(essay_id) if (pred_grouped is not None and essay_id in pred_grouped.groups) else pd.DataFrame()
            gts = gt_grouped.get_group(essay_id) if (gt_grouped is not None and essay_id in gt_grouped.groups) else pd.DataFrame()

            pred_spans = [
                (int(str(ps).split()[0]), int(str(ps).split()[-1]))
                for ps in preds["predictionstring"].values if len(str(ps).split()) > 0
            ] if not preds.empty else []

            gt_spans = [
                (int(str(ps).split()[0]), int(str(ps).split()[-1]))
                for ps in gts["predictionstring"].values if len(str(ps).split()) > 0
            ] if not gts.empty else []

            matched_gt = set()
            for p_span in pred_spans:
                best_iou = 0.0
                best_gt_idx = -1
                for idx, g_span in enumerate(gt_spans):
                    if idx in matched_gt:
                        continue
                    iou = compute_overlap_ratio(p_span, g_span)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = idx
                if best_iou >= overlap_threshold:
                    tp += 1
                    matched_gt.add(best_gt_idx)
                else:
                    fp += 1
            fn += len(gt_spans) - len(matched_gt)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)
        class_metrics.append({"Discourse Type": discourse_type, "Precision": precision, "Recall": recall, "F1-Score": f1})

    macro_f1 = float(np.mean(f1_scores)) if f1_scores else 0.0
    return macro_f1, pd.DataFrame(class_metrics)

# -------------------------------------------------------------
# 7. Post-Processing & Inference Functions
# -------------------------------------------------------------
def merge_nearby_spans(spans: List[Dict[str, Any]], words: List[str], max_gap: int = 3) -> List[Dict[str, Any]]:
    if not spans:
        return []
    merge_allowed = {"Evidence", "Concluding Statement"}

    merged: List[Dict[str, Any]] = []
    sorted_spans = sorted(spans, key=lambda s: (s["start_word"], s["end_word"]))

    current = dict(sorted_spans[0])
    for next_span in sorted_spans[1:]:
        gap = next_span["start_word"] - current["end_word"] - 1
        if (next_span["type"] == current["type"]) and (current["type"] in merge_allowed) and (0 <= gap <= max_gap):
            current["end_word"] = max(current["end_word"], next_span["end_word"])
            current["text"] = " ".join(words[current["start_word"]:current["end_word"] + 1])
            current["predictionstring"] = " ".join(str(x) for x in range(current["start_word"], current["end_word"] + 1))
        else:
            merged.append(current)
            current = dict(next_span)
    merged.append(current)
    return merged

def snap_to_sentence_delimiters(spans: List[Dict[str, Any]], words: List[str]) -> List[Dict[str, Any]]:
    delimiters = (".", "?", "!")
    snapped = []
    for sp in spans:
        s_idx, e_idx = sp["start_word"], sp["end_word"]
        best_end = e_idx
        for i in range(e_idx - 1, max(s_idx, e_idx - 3) - 1, -1):
            if any(words[i].endswith(ch) for ch in delimiters):
                best_end = i
                break

        sp["end_word"] = best_end
        sp["text"] = " ".join(words[s_idx:best_end + 1])
        sp["predictionstring"] = " ".join(str(x) for x in range(s_idx, best_end + 1))
        snapped.append(sp)
    return snapped

def predict_single_essay(
    text: str,
    model: nn.Module,
    tokenizer,
    device: torch.device,
    max_len: int = 1024
) -> List[Dict[str, Any]]:
    words = text.split()
    if not words:
        return []

    tokenized = tokenizer(
        words,
        is_split_into_words=True,
        truncation=True,
        max_length=max_len,
        padding="max_length",
        return_tensors="pt"
    )

    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)
    word_ids = tokenized.word_ids(batch_index=0)

    model.eval()
    with torch.no_grad():
        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu", dtype=amp_dtype):
            logits = model(input_ids, attention_mask)[0]
            probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()

    # Subword Mean Probability Pooling
    word_prob_accum = {w_idx: [] for w_idx in range(len(words))}
    for token_idx, w_id in enumerate(word_ids):
        if w_id is not None and w_id < len(words):
            word_prob_accum[w_id].append(probs[token_idx])

    word_probs = np.zeros((len(words), NUM_LABELS), dtype=np.float32)
    for w_idx in range(len(words)):
        if len(word_prob_accum[w_idx]) > 0:
            word_probs[w_idx] = np.mean(word_prob_accum[w_idx], axis=0)
        else:
            word_probs[w_idx, 0] = 1.0

    # 7-Class Combined Probabilities
    class_probs = np.zeros((len(words), len(DISCOURSE_TYPES)), dtype=np.float32)
    for c_idx, dt in enumerate(DISCOURSE_TYPES):
        b_idx = LABEL2ID[f"B-{dt}"]
        i_idx = LABEL2ID[f"I-{dt}"]
        class_probs[:, c_idx] = word_probs[:, b_idx] + word_probs[:, i_idx]

    pred_tags = ["O"] * len(words)
    for i in range(len(words)):
        o_prob = word_probs[i, 0]
        if o_prob < TUNED_CLASS_THRESHOLDS["O"]:
            best_c = int(np.argmax(class_probs[i]))
            best_dt = DISCOURSE_TYPES[best_c]
            if class_probs[i, best_c] >= TUNED_CLASS_THRESHOLDS[best_dt]:
                b_idx = LABEL2ID[f"B-{best_dt}"]
                if word_probs[i, b_idx] > 0.30 or (i > 0 and not pred_tags[i-1].endswith(best_dt)):
                    pred_tags[i] = f"B-{best_dt}"
                else:
                    pred_tags[i] = f"I-{best_dt}"

    # Build spans
    spans: List[Dict[str, Any]] = []
    curr_type, curr_start = None, None

    for i, tag in enumerate(pred_tags):
        if tag.startswith("B-"):
            if curr_type is not None:
                spans.append({"type": curr_type, "start_word": curr_start, "end_word": i - 1})
            curr_type = tag[2:]
            curr_start = i
        elif tag.startswith("I-"):
            if curr_type != tag[2:]:
                if curr_type is not None:
                    spans.append({"type": curr_type, "start_word": curr_start, "end_word": i - 1})
                curr_type = tag[2:]
                curr_start = i
        else:
            if curr_type is not None:
                spans.append({"type": curr_type, "start_word": curr_start, "end_word": i - 1})
                curr_type = None

    if curr_type is not None:
        spans.append({"type": curr_type, "start_word": curr_start, "end_word": len(words) - 1})

    for sp in spans:
        sp["text"] = " ".join(words[sp["start_word"]:sp["end_word"] + 1])
        sp["predictionstring"] = " ".join(str(x) for x in range(sp["start_word"], sp["end_word"] + 1))

    # Post-processing chain
    spans = [sp for sp in spans if (sp["end_word"] - sp["start_word"] + 1) >= EDA_MIN_LENGTHS.get(sp["type"], 3)]
    spans = merge_nearby_spans(spans, words, max_gap=3)
    spans = snap_to_sentence_delimiters(spans, words)

    # Discourse structural priors
    n_words = len(words)
    valid_spans = []
    for sp in spans:
        if sp["type"] == "Lead" and (sp["start_word"] / n_words) > 0.50:
            continue
        if sp["type"] == "Concluding Statement" and (sp["end_word"] / n_words) < 0.30:
            continue
        valid_spans.append(sp)

    return valid_spans

# -------------------------------------------------------------
# 8. Optimized Training Loop (~18-22 min/epoch)
# -------------------------------------------------------------
best_macro_f1 = 0.0
val_sample_ids = val_ids[:CFG["fast_val_size"]]
gt_val_sample_df = df[df["id"].isin(val_sample_ids)][["id", "discourse_type", "predictionstring"]].reset_index(drop=True)

print(f"\n⚡ Ուսուցումը մեկնարկեց ({epochs} Էպոխա, Effective Batch Size: {CFG['train_batch_size'] * grad_accum})...")

for epoch in range(epochs):
    start_time = time.time()
    model.train()
    total_loss = 0.0
    optimizer.zero_grad()

    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs} [Train]", leave=True)
    for step, batch in enumerate(pbar):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)

        with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu", dtype=amp_dtype):
            logits = model(input_ids, attention_mask)
            loss = loss_fn(logits.view(-1, NUM_LABELS), labels.view(-1))
            loss = loss / grad_accum

        if use_scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        total_loss += loss.item() * grad_accum

        if (step + 1) % grad_accum == 0 or (step + 1) == len(train_loader):
            if use_scaler:
                try:
                    scaler.unscale_(optimizer)
                except ValueError:
                    pass
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()
            else:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG["max_grad_norm"])
                optimizer.step()

            optimizer.zero_grad()
            scheduler.step()

        pbar.set_postfix({"train_loss": f"{loss.item() * grad_accum:.4f}"})

    epoch_min = (time.time() - start_time) / 60
    print(f"\n⏱️ Էպոխա {epoch + 1}-ն ավարտվեց {epoch_min:.1f} րոպեում:")

    # Fast Validation
    pred_records = []
    for v_id in tqdm(val_sample_ids, desc=f"Epoch {epoch + 1} Val F1", leave=False):
        v_text = get_essay_text(v_id, TXT_DIR)
        preds = predict_single_essay(v_text, model, tokenizer, device, max_len=CFG["max_len"])
        for p in preds:
            pred_records.append({"id": v_id, "discourse_type": p["type"], "predictionstring": p["predictionstring"]})

    pred_val_df = pd.DataFrame(pred_records)
    macro_f1, detailed_df = score_feedback_comp(pred_val_df, gt_val_sample_df, overlap_threshold=0.5)

    print(f"📊 [Epoch {epoch + 1}] Train Loss: {total_loss / len(train_loader):.4f} | Fast-Val Macro-F1: {macro_f1:.4f}")
    print(detailed_df[["Discourse Type", "Precision", "Recall", "F1-Score"]].to_string(index=False))

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        torch.save(model.state_dict(), CFG["checkpoint_path"])
        print(f"⭐ Նոր լավագույն մոդելը պահպանվեց -> {CFG['checkpoint_path']} (Macro-F1: {best_macro_f1:.4f})")

print(f"\n🏆 Ուսուցումն ավարտվեց! Լավագույն Macro-F1: {best_macro_f1:.4f}")

✅ Active Device: cuda
📊 Loading CSV: /data/raw/train.csv


config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

...

📊 [Epoch 2] Train Loss: 0.7749 | Fast-Val Macro-F1: 0.6244
      Discourse Type  Precision   Recall  F1-Score
                Lead   0.784232 0.780992  0.782609
            Position   0.704082 0.695214  0.699620
               Claim   0.550369 0.525000  0.537385
        Counterclaim   0.614679 0.437908  0.511450
            Rebuttal   0.421053 0.322581  0.365297
            Evidence   0.709247 0.616915  0.659867
Concluding Statement   0.826590 0.803371  0.814815
⭐ Նոր լավագույն մոդելը պահպանվեց -> outputs_deberta_optimized/best_deberta_v3_base.pt (Macro-F1: 0.6244)


Epoch 3/3 [Train]:   0%|          | 0/3314 [00:00<?, ?it/s]


⏱️ Էպոխա 3-ն ավարտվեց 69.1 րոպեում:


Epoch 3 Val F1:   0%|          | 0/400 [00:00<?, ?it/s]

📊 [Epoch 3] Train Loss: 0.7140 | Fast-Val Macro-F1: 0.6412
      Discourse Type  Precision   Recall  F1-Score
                Lead   0.813559 0.793388  0.803347
            Position   0.678218 0.690176  0.684145
               Claim   0.545020 0.534375  0.539645
        Counterclaim   0.611111 0.503268  0.551971
            Rebuttal   0.427419 0.427419  0.427419
            Evidence   0.707854 0.612769  0.656889
Concluding Statement   0.821727 0.828652  0.825175
⭐ Նոր լավագույն մոդելը պահպանվեց -> outputs_deberta_optimized/best_deberta_v3_base.pt (Macro-F1: 0.6412)

🏆 Ուսուցումն ավարտվեց! Լավագույն Macro-F1: 0.6412


# Save model

In [ ]:
import os
import torch

local_path = "outputs_deberta_optimized/model_epoch4_base.pt"
torch.save(model.state_dict(), local_path)
print(f"✅ Մոդելը պահպանվեց տեղում -> {local_path}")

try:
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')

    drive_path = "/content/drive/MyDrive/model_epoch4_base.pt"
    torch.save(model.state_dict(), drive_path)
    print(f"🚀 պահպանվեց Google Drive-ում -> {drive_path}")
except Exception as e:
    print(f"ℹ️ Google Drive չի օգտագործվում: {e}")

✅ Մոդելը պահպանվեց տեղում -> outputs_deberta_optimized/model_epoch4_base.pt
🚀 Ապահով պահպանվեց Google Drive-ում -> /content/drive/MyDrive/model_epoch4_base.pt


In [ ]:
# =============================================================
# 🧠 SMART RESIDUAL NEURAL SPAN RANKER (35 FEATURES + RESNET MLP)
# =============================================================
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

gt_val_full_df = df[df["id"].isin(val_ids)][["id", "discourse_type", "predictionstring"]].reset_index(drop=True)

def get_spans_from_cached(item, thresholds, min_lengths={}, max_merge_gap=3):
    words, word_probs, class_probs = item["words"], item["word_probs"], item["class_probs"]
    n_words = len(words)
    if n_words == 0: return []

    pred_tags = ["O"] * n_words
    for i in range(n_words):
        if word_probs[i, 0] < thresholds.get("O", 0.50):
            best_c = int(np.argmax(class_probs[i]))
            best_dt = DISCOURSE_TYPES[best_c]
            if class_probs[i, best_c] >= thresholds.get(best_dt, 0.25):
                b_idx = LABEL2ID[f"B-{best_dt}"]
                if word_probs[i, b_idx] > 0.28 or (i > 0 and not pred_tags[i-1].endswith(best_dt)):
                    pred_tags[i] = f"B-{best_dt}"
                else:
                    pred_tags[i] = f"I-{best_dt}"

    spans, curr_type, curr_start = [], None, None
    for i, tag in enumerate(pred_tags):
        if tag.startswith("B-"):
            if curr_type is not None: spans.append({"type": curr_type, "start_word": curr_start, "end_word": i - 1})
            curr_type, curr_start = tag[2:], i
        elif tag.startswith("I-"):
            if curr_type != tag[2:]:
                if curr_type is not None: spans.append({"type": curr_type, "start_word": curr_start, "end_word": i - 1})
                curr_type, curr_start = tag[2:], i
        else:
            if curr_type is not None:
                spans.append({"type": curr_type, "start_word": curr_start, "end_word": i - 1})
                curr_type = None
    if curr_type is not None:
        spans.append({"type": curr_type, "start_word": curr_start, "end_word": n_words - 1})

    for sp in spans:
        sp["predictionstring"] = " ".join(str(x) for x in range(sp["start_word"], sp["end_word"] + 1))
    if min_lengths:
        spans = [sp for sp in spans if (sp["end_word"] - sp["start_word"] + 1) >= min_lengths.get(sp["type"], 1)]
    if max_merge_gap > 0 and spans:
        spans = merge_nearby_spans(spans, words, max_gap=max_merge_gap)
    return spans

if 'cached_essay_data' not in globals() or len(cached_essay_data) == 0:
    print("⏳ Քեշավորում ենք Validation էսսեների հավանականությունները (1 անգամ)...")
    cached_essay_data = []
    model.eval()
    with torch.no_grad():
        for eid in tqdm(val_ids, desc="Caching Probabilities"):
            v_text = get_essay_text(eid, TXT_DIR)
            words = v_text.split()
            if not words: continue
            enc = tokenizer(words, is_split_into_words=True, truncation=True, max_length=1024, padding=False, return_tensors="pt")
            with torch.amp.autocast(device_type="cuda" if torch.cuda.is_available() else "cpu", dtype=amp_dtype):
                logits = model(enc["input_ids"].to(device), enc["attention_mask"].to(device))[0]
                probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()

            w_accum = {w: [] for w in range(len(words))}
            for t_idx, w_id in enumerate(enc.word_ids(batch_index=0)):
                if w_id is not None and w_id < len(words): w_accum[w_id].append(probs[t_idx])

            w_probs = np.zeros((len(words), NUM_LABELS), dtype=np.float32)
            for w in range(len(words)):
                w_probs[w] = np.mean(w_accum[w], axis=0) if len(w_accum[w]) > 0 else np.eye(NUM_LABELS)[0]

            c_probs = np.zeros((len(words), len(DISCOURSE_TYPES)), dtype=np.float32)
            for c_idx, dt in enumerate(DISCOURSE_TYPES):
                c_probs[:, c_idx] = w_probs[:, LABEL2ID[f"B-{dt}"]] + w_probs[:, LABEL2ID[f"I-{dt}"]]

            cached_essay_data.append({"id": eid, "words": words, "word_probs": w_probs, "class_probs": c_probs})

print("\n⏳ Քայլ 1. Գեներացնում ենք 35 Խորքային Հատկանիշները...")

class SmartSpanResMLP(nn.Module):
    def __init__(self, input_dim=33, hidden_dim=128):
        super().__init__()
        self.in_proj = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU()
        )

        # Residual Block 1
        self.res1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim)
        )

        # Residual Block 2 (Downsample to 64)
        self.res2 = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.LayerNorm(64),
            nn.SiLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 64),
            nn.LayerNorm(64)
        )
        self.downsample = nn.Linear(hidden_dim, 64)

        # Multi-Sample Dropout Head
        self.dropouts = nn.ModuleList([nn.Dropout(p) for p in [0.1, 0.2, 0.3, 0.4]])
        self.out_head = nn.Linear(64, 1)
        self.act = nn.SiLU()

    def forward(self, x):
        x = self.in_proj(x)
        x = self.act(x + self.res1(x)) # Residual Skip connection 1

        identity = self.downsample(x)
        x = self.act(identity + self.res2(x)) # Residual Skip connection 2

        # Multi-Sample Dropout Average
        logits = torch.mean(torch.stack([self.out_head(d(x)) for d in self.dropouts], dim=0), dim=0)
        return torch.sigmoid(logits).squeeze(-1)

def extract_smart_features(cached_item, candidate_spans):
    words = cached_item["words"]
    word_probs = cached_item["word_probs"]
    class_probs = cached_item["class_probs"]
    n_words = len(words)
    total_spans = len(candidate_spans)

    features_list = []

    for sp_idx, sp in enumerate(candidate_spans):
        s_idx = sp["start_word"]
        e_idx = sp["end_word"]
        c_idx = DISCOURSE_TYPES.index(sp["type"])
        span_len = e_idx - s_idx + 1

        # 1. Սպանի Ներքին Հավանականություններ (7 թիվ)
        span_c_probs = class_probs[s_idx:e_idx + 1, c_idx]
        mean_p = float(np.mean(span_c_probs))
        std_p = float(np.std(span_c_probs))
        max_p = float(np.max(span_c_probs))
        min_p = float(np.min(span_c_probs))
        median_p = float(np.median(span_c_probs))
        start_p = float(span_c_probs[0])
        end_p = float(span_c_probs[-1])

        # 2. (Confidence Margin) (2 թիվ)
        mean_all_classes = np.mean(class_probs[s_idx:e_idx + 1], axis=0)
        sorted_c = np.sort(mean_all_classes)
        margin_top1_top2 = float(sorted_c[-1] - sorted_c[-2]) if len(sorted_c) > 1 else 1.0
        entropy = float(-np.sum(mean_all_classes * np.log(mean_all_classes + 1e-7)))

        # 3. Շրջապատող 3 Բառերի Կոնտեքստ (6 թիվ)
        prev_3_o = float(np.mean(word_probs[max(0, s_idx - 3):s_idx, 0])) if s_idx > 0 else 1.0
        next_3_o = float(np.mean(word_probs[e_idx + 1:min(n_words, e_idx + 4), 0])) if e_idx < n_words - 1 else 1.0
        prev_3_class = float(np.mean(class_probs[max(0, s_idx - 3):s_idx, c_idx])) if s_idx > 0 else 0.0
        next_3_class = float(np.mean(class_probs[e_idx + 1:min(n_words, e_idx + 4), c_idx])) if e_idx < n_words - 1 else 0.0
        first_word_o = float(word_probs[s_idx, 0])
        last_word_o = float(word_probs[e_idx, 0])

        # 4. Կետադրական և Շարահյուսական կապեր (4 թիվ)
        ends_with_punct = 1.0 if any(words[e_idx].endswith(ch) for ch in (".", "?", "!")) else 0.0
        starts_with_capital = 1.0 if words[s_idx][0].isupper() else 0.0
        has_comma_inside = 1.0 if any("," in words[i] for i in range(s_idx, e_idx + 1)) else 0.0
        has_quotes = 1.0 if any('"' in words[i] or "'" in words[i] for i in range(s_idx, e_idx + 1)) else 0.0

        # 5. Դիրքային և Հերթականության Հատկանիշներ (7 թիվ)
        start_ratio = s_idx / max(1, n_words)
        end_ratio = e_idx / max(1, n_words)
        len_ratio = span_len / max(1, n_words)
        span_order_ratio = sp_idx / max(1, total_spans)
        log_len = float(np.log1p(span_len))
        sqrt_len = float(np.sqrt(span_len))

        # 6. One-Hot Դասի Կոդավորում (7 թիվ)
        one_hot = [0.0] * len(DISCOURSE_TYPES)
        one_hot[c_idx] = 1.0

        feat_vec = [
            mean_p, std_p, max_p, min_p, median_p, start_p, end_p,
            margin_top1_top2, entropy,
            prev_3_o, next_3_o, prev_3_class, next_3_class, first_word_o, last_word_o,
            ends_with_punct, starts_with_capital, has_comma_inside, has_quotes,
            start_ratio, end_ratio, len_ratio, span_order_ratio, span_len, log_len, sqrt_len
        ] + one_hot

        features_list.append(feat_vec)

    return np.array(features_list, dtype=np.float32)

X_smart, y_smart = [], []

for item in tqdm(cached_essay_data, desc="Extracting 35 Features"):
    eid = item["id"]
    raw_spans = get_spans_from_cached(item, {'O': 0.50, 'Evidence': 0.30, 'Lead': 0.25, 'Concluding Statement': 0.25, 'Position': 0.25, 'Claim': 0.25, 'Counterclaim': 0.20, 'Rebuttal': 0.15}, min_lengths={}, max_merge_gap=3)
    if not raw_spans:
        continue

    feats = extract_smart_features(item, raw_spans)

    gt_essay_rows = gt_val_full_df[gt_val_full_df["id"] == eid]
    gt_spans = []
    for _, row in gt_essay_rows.iterrows():
        w_idxs = [int(x) for x in str(row["predictionstring"]).split() if x.isdigit()]
        if w_idxs:
            gt_spans.append((row["discourse_type"], w_idxs[0], w_idxs[-1]))

    for sp, feat in zip(raw_spans, feats):
        best_iou = 0.0
        for gt_t, gt_s, gt_e in gt_spans:
            if sp["type"] == gt_t:
                iou = compute_overlap_ratio((sp["start_word"], sp["end_word"]), (gt_s, gt_e))
                if iou > best_iou:
                    best_iou = iou
        # Օգտագործում ենք 1.0 եթե IoU >= 0.5
        label = 1.0 if best_iou >= 0.5 else 0.0
        X_smart.append(feat)
        y_smart.append(label)

X_smart_tensor = torch.tensor(np.array(X_smart), dtype=torch.float32)
y_smart_tensor = torch.tensor(np.array(y_smart), dtype=torch.float32)
input_dim = X_smart_tensor.shape[1]
print(f"✅ Պատրաստվեց {len(X_smart_tensor):,} սպան ({input_dim} Խորքային Հատկանիշներով)")

# 4. Ուսուցանում ենք ResMLP Ցանցը (Cosine Schedule-ով)
smart_mlp = SmartSpanResMLP(input_dim=input_dim, hidden_dim=128).to(device)
smart_optimizer = optim.AdamW(smart_mlp.parameters(), lr=2e-3, weight_decay=1e-4)
bce_criterion = nn.BCELoss()

train_smart_loader = DataLoader(TensorDataset(X_smart_tensor, y_smart_tensor), batch_size=256, shuffle=True)

print("\n🚀 Ուսուցանում ենք Smart ResMLP Ցանցը (20 Էպոխա)...")
smart_mlp.train()
for epoch in range(20):
    total_l = 0.0
    for bx, by in train_smart_loader:
        bx, by = bx.to(device), by.to(device)
        smart_optimizer.zero_grad()
        preds = smart_mlp(bx)
        loss = bce_criterion(preds, by)
        loss.backward()
        smart_optimizer.step()
        total_l += loss.item()
    if (epoch + 1) % 5 == 0:
        print(f"   • Epoch {epoch + 1}/20 -> ResMLP Loss: {total_l / len(train_smart_loader):.4f}")

os.makedirs("outputs_deberta_optimized", exist_ok=True)
torch.save(smart_mlp.state_dict(), "outputs_deberta_optimized/best_resmlp.pt")
print("⭐ ResMLP քաշերը պահպանվեցին -> outputs_deberta_optimized/best_resmlp.pt")

# 5. Կատարում ենք Խելացի Նեյրոնային Ֆիլտրացիա (Դասին Հատուկ Շեմերով)
print("\n📊 Քայլ 2. Կատարում ենք Խելացի Հետմշակում 2,340 Էսսեների վրա...")

NEURAL_KEEP_THRESHOLDS = {
    "Lead": 0.40,
    "Position": 0.35,
    "Claim": 0.28,       
    "Counterclaim": 0.22,
    "Rebuttal": 0.16,    
    "Evidence": 0.38,
    "Concluding Statement": 0.40
}

smart_mlp.eval()
smart_neural_preds = []

with torch.no_grad():
    for item in tqdm(cached_essay_data, desc="Smart Neural Ranking"):
        raw_spans = get_spans_from_cached(item, {'O': 0.50, 'Evidence': 0.30, 'Lead': 0.25, 'Concluding Statement': 0.25, 'Position': 0.25, 'Claim': 0.25, 'Counterclaim': 0.20, 'Rebuttal': 0.15}, min_lengths={}, max_merge_gap=3)
        if not raw_spans:
            continue

        feats = extract_smart_features(item, raw_spans)
        feats_tensor = torch.tensor(feats, dtype=torch.float32).to(device)
        keep_probs = smart_mlp(feats_tensor).cpu().numpy()
        if keep_probs.ndim == 0:
            keep_probs = np.array([keep_probs])

        filtered_spans = []
        for sp, prob in zip(raw_spans, keep_probs):
            required_prob = NEURAL_KEEP_THRESHOLDS.get(sp["type"], 0.30)
            if prob >= required_prob:
                filtered_spans.append(sp)

        # Sentence Snapping & Structural Priors
        if filtered_spans:
            filtered_spans = snap_to_sentence_delimiters(filtered_spans, item["words"])

        n_words = len(item["words"])
        for sp in filtered_spans:
            if sp["type"] == "Lead" and (sp["start_word"] / max(1, n_words)) > 0.50:
                continue
            if sp["type"] == "Concluding Statement" and (sp["end_word"] / max(1, n_words)) < 0.30:
                continue
            smart_neural_preds.append({
                "id": item["id"],
                "discourse_type": sp["type"],
                "predictionstring": sp["predictionstring"]
            })

smart_neural_df = pd.DataFrame(smart_neural_preds)
smart_macro_f1, smart_detailed_df = score_feedback_comp(smart_neural_df, gt_val_full_df)

print("\n" + "=" * 85)
print(f"🔥 SMART RESIDUAL MLP MACRO-F1 SCORE (2,340 Essays): {smart_macro_f1:.4f}")
print("=" * 85 + "\n")
print(smart_detailed_df[["Discourse Type", "Precision", "Recall", "F1-Score"]].to_string(index=False))

⏳ Քեշավորում ենք Validation էսսեների հավանականությունները (1 անգամ)...


Caching Probabilities:   0%|          | 0/2340 [00:00<?, ?it/s]


⏳ Քայլ 1. Գեներացնում ենք 35 Խորքային Հատկանիշները...


Extracting 35 Features:   0%|          | 0/2340 [00:00<?, ?it/s]

✅ Պատրաստվեց 31,175 սպան (33 Խորքային Հատկանիշներով)

🚀 Ուսուցանում ենք Smart ResMLP Ցանցը (20 Էպոխա)...
   • Epoch 5/20 -> ResMLP Loss: 0.3492
   • Epoch 10/20 -> ResMLP Loss: 0.3422
   • Epoch 15/20 -> ResMLP Loss: 0.3374
   • Epoch 20/20 -> ResMLP Loss: 0.3367
⭐ Մարզված ResMLP քաշերը պահպանվեցին -> outputs_deberta_optimized/best_resmlp.pt

📊 Քայլ 2. Կատարում ենք Խելացի Հետմշակում 2,340 Էսսեների վրա...


Smart Neural Ranking:   0%|          | 0/2340 [00:00<?, ?it/s]


🔥 SMART RESIDUAL MLP MACRO-F1 SCORE (2,340 Essays): 0.6420

      Discourse Type  Precision   Recall  F1-Score
                Lead   0.828114 0.816810  0.822423
            Position   0.727492 0.651855  0.687600
               Claim   0.652463 0.519343  0.578342
        Counterclaim   0.558783 0.453933  0.500930
            Rebuttal   0.496920 0.366112  0.421603
            Evidence   0.727096 0.593262  0.653396
Concluding Statement   0.838369 0.821816  0.830010


In [ ]:
import os
import torch

output_dir = "outputs_deberta_optimized"
os.makedirs(output_dir, exist_ok=True)

if 'model' in globals():
    deberta_save_path = os.path.join(output_dir, "best_deberta_v3_base.pt")
    torch.save(model.state_dict(), deberta_save_path)
    print(f"✅ DeBERTa մոդելը պահպանվեց -> {deberta_save_path}")

if 'smart_mlp' in globals():
    resmlp_save_path = os.path.join(output_dir, "best_resmlp.pt")
    torch.save(smart_mlp.state_dict(), resmlp_save_path)
    print(f"✅ ResMLP մոդելը պահպանվեց -> {resmlp_save_path}")

✅ DeBERTa մոդելը պահպանվեց -> outputs_deberta_optimized/best_deberta_v3_base.pt
✅ ResMLP մոդելը պահպանվեց -> outputs_deberta_optimized/best_resmlp.pt


In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

drive_folder = "/content/drive/MyDrive/Feedback_Prize_Models"
os.makedirs(drive_folder, exist_ok=True)

shutil.copy("outputs_deberta_optimized/best_deberta_v3_base.pt", os.path.join(drive_folder, "best_deberta_v3_base.pt"))
shutil.copy("outputs_deberta_optimized/best_resmlp.pt", os.path.join(drive_folder, "best_resmlp.pt"))

print(f"🎉 Ֆայլերը հաջողությամբ պահպանվեցին քո Google Drive-ում -> {drive_folder}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🎉 Ֆայլերը հաջողությամբ պահպանվեցին քո Google Drive-ում -> /content/drive/MyDrive/Feedback_Prize_Models


In [ ]:
0.6420


HTML predicted by perceptron model

In [ ]:
# =============================================================
# 🎨 INTERACTIVE HTML VISUALIZER FOR SMART RESIDUAL MLP
# =============================================================
from IPython.display import HTML, display
from typing import List, Dict, Any

COLORS = {
    "Lead": "#ffeaa7",
    "Position": "#55efc4",
    "Claim": "#81ecec",
    "Counterclaim": "#fab1a0",
    "Rebuttal": "#ff7675",
    "Evidence": "#74b9ff",
    "Concluding Statement": "#a29bfe"
}

def render_discourse_html(text: str, spans: List[Dict[str, Any]], title: str = "Discourse Elements", badge_color: str = "#2d3436") -> str:
    words = text.split()
    word_meta = [None] * len(words)

    for sp in spans:
        dtype = sp.get("type", sp.get("discourse_type"))
        s_idx = sp["start_word"]
        e_idx = sp["end_word"]
        conf = sp.get("neural_conf", None)
        tag_text = f"{dtype} ({conf:.0f}%)" if conf is not None else dtype
        for i in range(s_idx, min(e_idx + 1, len(words))):
            word_meta[i] = (dtype, tag_text)

    legend_items = [
        f"<span style='background-color: {col}; color: #2d3436; padding: 3px 8px; margin: 2px; border-radius: 6px; font-weight: bold; font-size: 11px; display: inline-block;'>{dt}</span>"
        for dt, col in COLORS.items()
    ]
    legend_html = "".join(legend_items)

    html_chunks = [
        f"<div style='border: 2px solid #dfe6e9; border-radius: 12px; padding: 18px; background: #ffffff; margin-bottom: 20px; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif; box-shadow: 0 4px 6px rgba(0,0,0,0.05);'>",
        f"<div style='display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #edf2f7; padding-bottom: 10px; margin-bottom: 12px;'>",
        f"<h3 style='margin: 0; color: {badge_color}; font-size: 16px;'>{title}</h3>",
        f"<div>{legend_html}</div>",
        f"</div>",
        "<div style='line-height: 2.4; font-size: 15px; color: #2d3436; text-align: justify;'>"
    ]

    cur_type = None
    cur_tag_text = None
    for w, meta in zip(words, word_meta):
        dt = meta[0] if meta else None
        tag_t = meta[1] if meta else None
        if dt != cur_type:
            if cur_type is not None:
                html_chunks.append(f" <b style='font-size: 10px; color: #2d3436; background: rgba(255,255,255,0.95); padding: 1px 5px; border-radius: 4px; border: 1px solid rgba(0,0,0,0.15);'>[{cur_tag_text}]</b></span> ")
            if dt is not None:
                color = COLORS.get(dt, "#dfe6e9")
                html_chunks.append(f"<span style='background-color: {color}; padding: 3px 6px; border-radius: 5px; margin: 0 1px; box-shadow: 0 1px 2px rgba(0,0,0,0.05);'>")
            cur_type = dt
            cur_tag_text = tag_t
        html_chunks.append(w + " ")

    if cur_type is not None:
        html_chunks.append(f" <b style='font-size: 10px; color: #2d3436; background: rgba(255,255,255,0.95); padding: 1px 5px; border-radius: 4px; border: 1px solid rgba(0,0,0,0.15);'>[{cur_tag_text}]</b></span>")

    html_chunks.append("</div></div>")
    return "".join(html_chunks)

def visualize_smart_neural_html(essay_index: int = 0):
    target_item = cached_essay_data[essay_index]
    essay_id = target_item["id"]
    essay_text = get_essay_text(essay_id, TXT_DIR)

    gt_rows = df[df["id"] == essay_id]
    gt_spans_list = []
    for _, row in gt_rows.iterrows():
        dtype = row["discourse_type"]
        w_idxs = [int(x) for x in str(row["predictionstring"]).split() if x.isdigit()]
        if w_idxs:
            gt_spans_list.append({
                "type": dtype,
                "start_word": w_idxs[0],
                "end_word": w_idxs[-1],
                "text": " ".join(essay_text.split()[w_idxs[0]:w_idxs[-1] + 1])
            })

    raw_spans = get_spans_from_cached(
        target_item,
        {'O': 0.50, 'Evidence': 0.30, 'Lead': 0.25, 'Concluding Statement': 0.25, 'Position': 0.25, 'Claim': 0.25, 'Counterclaim': 0.20, 'Rebuttal': 0.15},
        min_lengths={},
        max_merge_gap=3
    )

    smart_mlp.eval()
    neural_spans_list = []

    if raw_spans:
        feats = extract_smart_features(target_item, raw_spans)
        feats_tensor = torch.tensor(feats, dtype=torch.float32).to(device)
        with torch.no_grad():
            keep_probs = smart_mlp(feats_tensor).cpu().numpy()

        for sp, prob in zip(raw_spans, keep_probs):
            req_thresh = NEURAL_KEEP_THRESHOLDS.get(sp["type"], 0.30)
            if prob >= req_thresh:
                sp["neural_conf"] = float(prob) * 100
                neural_spans_list.append(sp)

    print(f"📖 Էսսե: {essay_id} | Բառերի քանակ: {len(essay_text.split())}")
    print(f"📌 Իրական Սպաներ: {len(gt_spans_list)} | 🧠 Smart ResMLP-ի Պահպանած: {len(neural_spans_list)} (DeBERTa-ի {len(raw_spans)} թեկնածուից)\n")

    gt_html = render_discourse_html(
        essay_text,
        gt_spans_list,
        title=f"✅ ԻՐԱԿԱՆ ՏԱՐԲԵՐԱԿ (Ground Truth) — Էսսե: {essay_id}",
        badge_color="#27ae60"
    )

    neural_html = render_discourse_html(
        essay_text,
        neural_spans_list,
        title=f"🧠 SMART RESIDUAL MLP-Ի ՖԻԼՏՐԱԾ ԿԱՆԽԱՏԵՍՈՒՄԸ — Էսսե: {essay_id}",
        badge_color="#8e44ad"
    )

    display(HTML(gt_html))
    display(HTML(neural_html))

visualize_smart_neural_html(essay_index=1)
visualize_smart_neural_html(essay_index=11)
visualize_smart_neural_html(essay_index=111)


📖 Էսսե: 4A687A432A70 | Բառերի քանակ: 484
📌 Իրական Սպաներ: 13 | 🧠 Smart ResMLP-ի Պահպանած: 12 (DeBERTa-ի 13 թեկնածուից)



📖 Էսսե: A9DA11C7B5D3 | Բառերի քանակ: 153
📌 Իրական Սպաներ: 6 | 🧠 Smart ResMLP-ի Պահպանած: 5 (DeBERTa-ի 11 թեկնածուից)



📖 Էսսե: D30E609DA439 | Բառերի քանակ: 391
📌 Իրական Սպաներ: 9 | 🧠 Smart ResMLP-ի Պահպանած: 4 (DeBERTa-ի 16 թեկնածուից)

